# 1. Model Leakage Demo

## Objetivo
Mostrar la diferencia entre un flujo con fuga de datos y un flujo sin fuga usando el dataset `heart.csv`.

## Idea del notebook
1. Cargar el dataset.
2. Trabajar primero con variables numéricas para aislar el efecto de la fuga.
3. Crear una variable artificial que introduce fuga de información.
4. Entrenar un modelo SVC con un flujo incorrecto.
5. Repetir el entrenamiento con un flujo más correcto, sin exponer el conjunto de prueba antes de tiempo.
6. Comparar los resultados con AUC.

> En el siguiente notebook se hará la versión completa y segura con `Pipeline` + `GridSearchCV`.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score

In [2]:
df = pd.read_csv("../heart.csv")
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [3]:
print("Dimensiones del dataset:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())
print("\nValores nulos por columna:")
print(df.isnull().sum())

Dimensiones del dataset: (918, 12)

Columnas:
['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS', 'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope', 'HeartDisease']

Valores nulos por columna:
Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64


## Selección de variables para esta demostración

En este primer notebook usaremos solo variables numéricas para que la demostración de fuga sea más clara y parecida al ejemplo del PDF.

Las variables categóricas se trabajarán correctamente en el segundo notebook usando `Pipeline`.

In [4]:
numeric_cols = [
    "Age",
    "RestingBP",
    "Cholesterol",
    "FastingBS",
    "MaxHR",
    "Oldpeak"
]

X = df[numeric_cols].copy()
y = df["HeartDisease"].copy()

print("Variables numéricas usadas:", numeric_cols)
print("Forma de X:", X.shape)
print("Forma de y:", y.shape)

Variables numéricas usadas: ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']
Forma de X: (918, 6)
Forma de y: (918,)


In [5]:
y.value_counts(normalize=True)

HeartDisease
1    0.553377
0    0.446623
Name: proportion, dtype: float64

## Flujo con fuga de datos

Aquí vamos a introducir una variable artificial llamada `leaky_feature`, construida a partir de la variable objetivo `HeartDisease`.

Esto está mal a propósito, porque hace que el modelo vea información que en la realidad no debería conocer.

In [6]:
np.random.seed(42)

X_leaky = X.copy()
X_leaky["leaky_feature"] = y + np.random.normal(0, 0.01, size=len(y))

X_leaky.head()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,leaky_feature
0,40,140,289,0,172,0.0,0.004967
1,49,160,180,0,156,1.0,0.998617
2,37,130,283,0,98,0.0,0.006477
3,48,138,214,0,108,1.5,1.015230
4,54,150,195,0,122,0.0,-0.002342


In [21]:
param_grid = {
    "C": [0.01, 0.1, 1, 10],
    "gamma": [0.01, 0.1, 1]
}

In [22]:
scaler = MinMaxScaler()
X_scaled_leaky = scaler.fit_transform(X_leaky)

X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(
    X_scaled_leaky,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

grid_1 = GridSearchCV(
    estimator=SVC(probability=True),
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc"
)

grid_1.fit(X_train_1, y_train_1)

y_scores_1 = grid_1.predict_proba(X_test_1)[:, 1]
auc_leak = roc_auc_score(y_test_1, y_scores_1)

print("AUC con fuga:", round(auc_leak, 4))
print("Mejores parámetros con fuga:", grid_1.best_params_)

AUC con fuga: 1.0
Mejores parámetros con fuga: {'C': 0.01, 'gamma': 0.01}


## Flujo sin fuga respecto al conjunto de prueba

Ahora repetimos el proceso sin usar la variable artificial y separando primero entrenamiento y prueba.

Después ajustamos el escalador solo con `X_train`.

In [17]:
X_clean = X.copy()

X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(
    X_clean,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = MinMaxScaler()
X_train_2_scaled = scaler.fit_transform(X_train_2)
X_test_2_scaled = scaler.transform(X_test_2)

grid_2 = GridSearchCV(
    estimator=SVC(probability=True),
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc"
)

grid_2.fit(X_train_2_scaled, y_train_2)

y_scores_2 = grid_2.predict_proba(X_test_2_scaled)[:, 1]
auc_no_leak = roc_auc_score(y_test_2, y_scores_2)

print("AUC sin fuga:", round(auc_no_leak, 4))
print("Mejores parámetros sin fuga:", grid_2.best_params_)

AUC sin fuga: 0.8881
Mejores parámetros sin fuga: {'C': 10, 'gamma': 1}


In [18]:
comparison = pd.DataFrame({
    "Escenario": ["Con fuga", "Sin fuga"],
    "AUC": [auc_leak, auc_no_leak]
})

comparison["AUC"] = comparison["AUC"].round(4)
comparison

,Escenario,AUC
0,Con fuga,1.0000
1,Sin fuga,0.8881


In [19]:
difference = auc_leak - auc_no_leak
print("Diferencia de AUC:", round(difference, 4))

Diferencia de AUC: 0.1119


In [20]:
if auc_leak > auc_no_leak:
    print("El flujo con fuga produce un resultado artificialmente más alto.")
else:
    print("Revisa la ejecución, porque el resultado no mostró el efecto esperado de la fuga.")

El flujo con fuga produce un resultado artificialmente más alto.


## Conclusión

- El flujo con fuga produce un AUC artificialmente alto.
- El flujo sin fuga entrega una evaluación más realista.
- La fuga ocurre porque el modelo termina viendo información que no debería conocer antes de evaluar.

Este notebook demuestra el problema.

En el siguiente notebook se construirá la solución correcta usando:
- `Pipeline`
- `GridSearchCV`
- varios modelos
- evaluación más completa